# End-to-End InSAR Sentinel-1 + ISCE Workflow (Consolidated)

This notebook replaces the original three notebooks:
1. Input preparation and cropping (former Part 1)
2. LSTM-based forecasting and score generation (former Part 2)
3. Geocoded output production (former Part 3)

Methodological principle: **the notebook should remain lightweight** (documentation + orchestration), while all reusable computational logic is implemented in `insar_pipeline/*.py`.

## 1) Path Configuration
Please adapt the following paths to your local computational environment.

In [ ]:
from pathlib import Path

BASE_INTERFEROGRAM_DIR = Path("/data6/WORKDIR/AmatriceSenDT22/merged/interferograms")
GEOM_REFERENCE_DIR = Path("/data6/WORKDIR/AmatriceSenDT22/merged/geom_reference")
NEXT_DATE = "20160821_20160902"

## 2) Stage-wise Execution (Recommended for Debugging and Analysis)

In [ ]:
from insar_pipeline import (
    CropConfig, batch_crop_filt_fine_cor,
    DatasetConfig, build_and_save_dataset,
    TrainingConfig, run_training_and_prediction,
    ScoreConfig, compute_and_save_score,
    OutputConfig, generate_geocoded_outputs,
)

## 2.1) Optional: StackSentinel input mode
If your input comes directly from ISCE stack pair folders, set `input_source='stack_int'` and choose whether to read ISCE `.cor` or compute coherence from `.int`.

In [ ]:
# Example configuration for stackSentinel input
from pathlib import Path
from insar_pipeline import DatasetConfig

STACK_DATASET_CONFIG = DatasetConfig(
    cropped_dir=BASE_INTERFEROGRAM_DIR / 'cropped',   # not used in stack_int mode
    output_dir=BASE_INTERFEROGRAM_DIR / 'cropped',
    input_source='stack_int',
    stack_root=BASE_INTERFEROGRAM_DIR,
    coherence_source='computed_phsig',  # 'isce' | 'computed_phsig' | 'computed_crlb'
    win=5,
    looks=25,
    std_thresh=1.0,
    use_circular_std=True,
    persist_computed_cor=False,
)

In [ ]:
# Step A: Crop coherence products and geometry rasters
cropped_outputs = batch_crop_filt_fine_cor(
    CropConfig(
        base_path=BASE_INTERFEROGRAM_DIR,
        geom_reference_path=GEOM_REFERENCE_DIR,
        output_base_path=BASE_INTERFEROGRAM_DIR / "cropped",
    )
)
print(f"Number of cropped files: {len(cropped_outputs)}")

In [ ]:
# Step B: Build training datasets (data.npy / data_std.npy / dates.pkl / geninue*.npy)
dataset_dir = build_and_save_dataset(
    DatasetConfig(
        cropped_dir=BASE_INTERFEROGRAM_DIR / "cropped",
        output_dir=BASE_INTERFEROGRAM_DIR / "cropped",
    )
)
print("Dataset directory:", dataset_dir)

In [ ]:
# Step C: Train model and generate prediction
predict_dir = run_training_and_prediction(
    TrainingConfig(
        dataset_dir=dataset_dir,
        output_dir=BASE_INTERFEROGRAM_DIR / "cropped",
        next_date=NEXT_DATE,
        epochs=15,
    )
)
print("Prediction directory:", predict_dir)

In [ ]:
# Step D: Compute normalized difference score (score.npy)
score_path = compute_and_save_score(
    ScoreConfig(
        dataset_dir=dataset_dir,
        predict_dir=predict_dir,
    )
)
print("Score file:", score_path)

In [ ]:
# Step E: Generate geocoded products and GeoTIFF outputs
output_files = generate_geocoded_outputs(
    OutputConfig(
        predict_dir=predict_dir,
        lat_file=BASE_INTERFEROGRAM_DIR / "cropped" / "lat_cropped.rdr",
        lon_file=BASE_INTERFEROGRAM_DIR / "cropped" / "lon_cropped.rdr",
    )
)
print(output_files)

## 3) One-Click Execution (Recommended After Validation)

In [ ]:
from insar_pipeline import run_full_pipeline

result = run_full_pipeline(
    base_dir=BASE_INTERFEROGRAM_DIR,
    geom_reference_dir=GEOM_REFERENCE_DIR,
    next_date=NEXT_DATE,
)
result

## 4) Suggested Extensions
- Externalize hyperparameters (e.g., epochs, batch size, spatial bounds) into a YAML configuration file.
- Standardize training logs in `predict/train.log` for experiment traceability.
- For larger-scale data, introduce sampling/chunking strategies in `modeling.py`.